 ==========================================================

 InsightForge AI

 Agentic AI for Intelligent Information Extraction

 Author : Sumit Yagik

 Version : 1.0.0

 ==========================================================

# InsightForge AI

## Objective

Build a production-inspired Agentic AI system capable of:

- Keyword Extraction
- Named Entity Recognition (NER)
- Intelligent Tool Selection
- Structured JSON Output
- Future RAG Integration

---

## Tech Stack

- Python
- Google Colab
- LangChain
- Groq
- KeyBERT
- spaCy
- Sentence Transformers

In [70]:
!pip install -qU \
langchain \
langchain-core \
langchain-community \
langchain-groq \
langgraph \
groq \
transformers==4.41.2 \
sentence-transformers==3.0.1 \
keybert==0.8.5 \
spacy \
faiss-cpu \
python-dotenv \
pydantic \
peft==0.11.1 \
tiktoken

In [22]:
# ==========================================================
# STANDARD LIBRARIES
# ==========================================================

import os
import warnings
from typing import Dict, List


# ==========================================================
# LANGCHAIN
# ==========================================================

from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage
from langchain_core.prompts import ChatPromptTemplate

# ==========================================================
# NLP
# ==========================================================

from sentence_transformers import SentenceTransformer
from keybert import KeyBERT
import spacy

# ==========================================================
# LLM
# ==========================================================

from langchain_groq import ChatGroq

# ==========================================================
# GOOGLE COLAB
# ==========================================================

from google.colab import userdata

In [23]:
# ==========================================================
# PROJECT CONFIGURATION
# ==========================================================

warnings.filterwarnings("ignore")

PROJECT_NAME = "InsightForge AI"
PROJECT_VERSION = "1.0.0"

EMBEDDING_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
SPACY_MODEL_NAME = "en_core_web_sm"
LLM_MODEL_NAME = "llama-3.3-70b-versatile"

In [24]:
#=============================================================================
# APPLICATION CONFIGURATION
#=============================================================================

class AppConfig:
  PROJECT_NAME = "InsightForge AI"
  PROJECT_VERSION = "1.0.0"
  EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
  SPACY_MODEL = "en_core_web_sm"
  LLM_MODEL = "llama-3.3-70b-versatile"
  MAX_KEYWORDS = 10

config = AppConfig()

In [25]:
# ==========================================================
# MODEL REGISTRY
# ==========================================================

class ModelRegistry:
    """
    Central registry for all AI models.
    Models are loaded lazily (only when needed).
    """

    def __init__(self):
        self.embedding_model = None
        self.keyword_model = None
        self.nlp = None
        self.llm = None
        self.summarizer = None

    #embedding model function
    def get_embedding_model(self):
      """
      Load the SentenceTransformer model only once.
      """

      if self.embedding_model is None:
        print("Loading Embedding Model...")

        self.embedding_model=SentenceTransformer(
            config.EMBEDDING_MODEL
        )
        print("Embedding Model Loaded Successfully!")
      else:
        print("Using Cached Embedding Model.")

      return self.embedding_model

    #keyword model function
    def get_keyword_model(self):
      """
      Load the KeyBERT model only once.
      """

      if self.keyword_model is None:
        print("Loading KeyBERT Model...")
        embedding_model = self.get_embedding_model()

        self.keyword_model = KeyBERT(
            model = keyword_model
            )
        print("KeyBERT Model Loaded Successfully!")

      else:
        print("Using Cached KeyBERT Model.")

      return self.keyword_model

    #NER tool function
    def get_spacy_model(self):
      """
      Load the spaCy model only once.
      """

      if self.nlp is None:

          print("Loading spaCy Model...")

          self.nlp = spacy.load(config.SPACY_MODEL)

          print("spaCy Model Loaded Successfully!")

      else:

          print("Using Cached spaCy Model.")

      return self.nlp


    #Summarization tool function
    def get_summarizer(self):

      if self.summarizer is None:

          print("Loading Summarizer...")

          self.summarizer = pipeline(
              "summarization",
              model="facebook/bart-large-cnn"
          )

      return self.summarizer

registry = ModelRegistry()


In [68]:
embedding_model = registry.get_embedding_model()

Using Cached Embedding Model.


In [71]:
keyword_model = registry.get_keyword_model()

Using Cached KeyBERT Model.


In [41]:
nlp = registry.get_spacy_model()

Loading spaCy Model...
spaCy Model Loaded Successfully!


In [42]:
print(dir(registry))

['__class__', '__delattr__', '__dict__', '__dir__', '__doc__', '__eq__', '__format__', '__ge__', '__getattribute__', '__getstate__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__le__', '__lt__', '__module__', '__ne__', '__new__', '__reduce__', '__reduce_ex__', '__repr__', '__setattr__', '__sizeof__', '__str__', '__subclasshook__', '__weakref__', 'embedding_model', 'get_embedding_model', 'get_keyword_model', 'get_spacy_model', 'get_summarizer', 'keyword_model', 'llm', 'nlp', 'summarizer']


In [43]:
import time

In [44]:
from langchain_core.tools import tool

In [45]:
@tool
def keyword_tool(text: str) -> dict:
    """
    Extract important keywords from text using KeyBERT.
    """
    if not isinstance(text, str):
        return {
            "status": "error",
            "tool_name": "keyword_tool",
            "execution_time": 0,
            "data": None,
            "metadata": None,
            "error": "Input must be a string."
        }

    if not text.strip():
        return {
            "status": "error",
            "tool_name": "keyword_tool",
            "execution_time": 0,
            "data": None,
            "metadata": None,
            "error": "Input text is empty."
        }

    # -----------------------------
    # Start Timer
    # -----------------------------
    start_time = time.time()

    # -----------------------------
    # Get KeyBERT Model
    # -----------------------------
    keyword_model = registry.get_keyword_model()

    # -----------------------------
    # Extract Keywords
    # -----------------------------
    keywords = keyword_model.extract_keywords(
        text,
        top_n=config.MAX_KEYWORDS,
        keyphrase_ngram_range=(1, 2),
        stop_words="english"
    )

    # -----------------------------
    # Convert Output
    # -----------------------------
    formatted_keywords = []

    for keyword, score in keywords:
        formatted_keywords.append(
            {
                "keyword": keyword,
                "score": round(float(score), 3)
            }
        )

    # -----------------------------
    # Stop Timer
    # -----------------------------
    #execution_time = round(time.time() - start_time, 3)

    # -----------------------------
    # Return Structured Response
    # -----------------------------
    return {
        "status": "success",
        "tool_name": "keyword_tool",
        "execution_time": round(time.time() - start_time, 3),

        "data": {
            "keywords": formatted_keywords
        },

        "metadata": {
            "keyword_count": len(formatted_keywords),
            "model": "KeyBERT"
        },

        "error": None
    }

In [46]:
keyword_tool.invoke({"text": "Deep learning is transforming image analysis."})

Loading KeyBERT Model...
Loading Embedding Model...


model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding Model Loaded Successfully!
KeyBERT Model Loaded Successfully!


{'status': 'success',
 'tool_name': 'keyword_tool',
 'execution_time': 7.437,
 'data': {'keywords': [{'keyword': 'learning transforming', 'score': 0.579},
   {'keyword': 'transforming image', 'score': 0.546},
   {'keyword': 'deep learning', 'score': 0.541},
   {'keyword': 'transforming', 'score': 0.51},
   {'keyword': 'image analysis', 'score': 0.446},
   {'keyword': 'learning', 'score': 0.371},
   {'keyword': 'image', 'score': 0.343},
   {'keyword': 'deep', 'score': 0.314},
   {'keyword': 'analysis', 'score': 0.271}]},
 'metadata': {'keyword_count': 9, 'model': 'KeyBERT'},
 'error': None}

In [47]:
keyword_tool.invoke({"text": ""})

{'status': 'error',
 'tool_name': 'keyword_tool',
 'execution_time': 0,
 'data': None,
 'metadata': None,
 'error': 'Input text is empty.'}

In [48]:
#Testing
sample_text = """
Deep learning has revolutionized computer vision.
Convolutional Neural Networks are widely used for image classification.
"""
result = keyword_tool.invoke(
    {
        "text": sample_text
    }
)

print(result)

Using Cached KeyBERT Model.
{'status': 'success', 'tool_name': 'keyword_tool', 'execution_time': 0.129, 'data': {'keywords': [{'keyword': 'deep learning', 'score': 0.577}, {'keyword': 'convolutional neural', 'score': 0.576}, {'keyword': 'vision convolutional', 'score': 0.565}, {'keyword': 'image classification', 'score': 0.496}, {'keyword': 'computer vision', 'score': 0.481}, {'keyword': 'convolutional', 'score': 0.467}, {'keyword': 'learning revolutionized', 'score': 0.453}, {'keyword': 'neural networks', 'score': 0.449}, {'keyword': 'classification', 'score': 0.386}, {'keyword': 'neural', 'score': 0.385}]}, 'metadata': {'keyword_count': 10, 'model': 'KeyBERT'}, 'error': None}


In [49]:
import time
from langchain_core.tools import tool

@tool
def ner_tool(text: str) -> dict:
    """
    Extract named entities from text using spaCy.
    """

    # Validation
    if not isinstance(text, str):
        return {
            "status": "error",
            "tool_name": "ner_tool",
            "execution_time": 0,
            "data": None,
            "metadata": None,
            "error": "Input must be a string."
        }

    if not text.strip():
        return {
            "status": "error",
            "tool_name": "ner_tool",
            "execution_time": 0,
            "data": None,
            "metadata": None,
            "error": "Input text is empty."
        }

    start_time = time.time()

    nlp = registry.get_spacy_model()

    doc = nlp(text)

    entities = []

    for ent in doc.ents:
        entities.append({
            "text": ent.text,
            "label": ent.label_
        })

    #execution_time = round(time.time() - start_time, 3)

    return {
        "status": "success",
        "tool_name": "ner_tool",
        "execution_time": round(time.time() - start_time, 3),
        "data": {
            "entities": entities
        },
        "metadata": {
            "entity_count": len(entities),
            "model": "spaCy"
        },
        "error": None
    }


In [50]:
# Testing
sample_text = """
Google CEO Sundar Pichai visited India and met Prime Minister Narendra Modi in New Delhi.
"""
result = ner_tool.invoke({
    "text": sample_text
})

print(result)

Using Cached spaCy Model.
{'status': 'success', 'tool_name': 'ner_tool', 'execution_time': 0.065, 'data': {'entities': [{'text': 'Google', 'label': 'ORG'}, {'text': 'Sundar Pichai', 'label': 'PERSON'}, {'text': 'India', 'label': 'GPE'}, {'text': 'Narendra Modi', 'label': 'PERSON'}, {'text': 'New Delhi', 'label': 'GPE'}]}, 'metadata': {'entity_count': 5, 'model': 'spaCy'}, 'error': None}


In [51]:
from transformers import pipeline

In [76]:
@tool
def summarize_tool(text: str) -> dict:
    """
    Generate a concise summary of the given text using the BART summarization model.
    This tool is useful for summarizing research papers, articles, reports,
    and other long documents while preserving the main ideas.
    """

    summarizer = registry.get_summarizer()

    input_words = len(text.split())

    max_len = min(120, max(30, input_words // 2))
    min_len = min(20, max_len - 5)

    summary = summarizer(
        text,
        max_length=max_len,
        min_length=min_len,
        do_sample=False
    )[0]["summary_text"]

    start_time = time.time()

    return {
      "status": "success",
      "tool_name": "summarize_tool",
      "execution_time": round(time.time() - start_time, 3),
      "data": {
          "summary": summary
      },
      "metadata": {
          "model": "facebook/bart-large-cnn"
      },
      "error": None
  }

In [53]:
# ==========================================================
# LOAD LLM
# ==========================================================

llm = ChatGroq(
    api_key=userdata.get("GROQ_API_KEY2"),
    model=config.LLM_MODEL,
    temperature=0
)

print("Groq LLM Loaded Successfully!")

Groq LLM Loaded Successfully!


In [54]:
tools = [
    keyword_tool,
    ner_tool,
    summarize_tool
]

print(tools)

[StructuredTool(name='keyword_tool', description='Extract important keywords from text using KeyBERT.', args_schema=<class 'langchain_core.utils.pydantic.keyword_tool'>, func=<function keyword_tool at 0x78a307a3e3e0>), StructuredTool(name='ner_tool', description='Extract named entities from text using spaCy.', args_schema=<class 'langchain_core.utils.pydantic.ner_tool'>, func=<function ner_tool at 0x78a307a3e520>), StructuredTool(name='summarize_tool', description='Generate a concise summary of the given text using the BART summarization model.\nThis tool is useful for summarizing research papers, articles, reports,\nand other long documents while preserving the main ideas.', args_schema=<class 'langchain_core.utils.pydantic.summarize_tool'>, func=<function summarize_tool at 0x78a2fc22e340>)]


In [35]:
import langchain
print(langchain.__version__)

1.3.11


In [55]:
!pip -q install langgraph

In [56]:
from langgraph.prebuilt import create_react_agent
from langchain_core.tools import tool

In [57]:
agent = create_react_agent(
    model=llm,
    tools=tools
)

In [58]:
response = agent.invoke(
    {
        "messages":[
            (
                "user",
                """
Extract keywords from this text.

Deep learning is widely used in computer vision and image segmentation.
"""
            )
        ]
    }
)

Using Cached KeyBERT Model.


In [59]:
print(response)

{'messages': [HumanMessage(content='\nExtract keywords from this text.\n\nDeep learning is widely used in computer vision and image segmentation.\n', additional_kwargs={}, response_metadata={}, id='bd45515c-f33a-421e-bbe9-ad949edf2a49'), AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'kk0b5kfd4', 'function': {'arguments': '{"text":"Deep learning is widely used in computer vision and image segmentation."}', 'name': 'keyword_tool'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 25, 'prompt_tokens': 412, 'total_tokens': 437, 'completion_time': 0.053373415, 'completion_tokens_details': None, 'prompt_time': 0.059372458, 'prompt_tokens_details': None, 'queue_time': 0.299069027, 'total_time': 0.112745873}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_3272ea2d91', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019f6a6a-0ad0-7f60-ac8c-6f29901b263f-0', tool

In [60]:
response = agent.invoke(
    {
        "messages":[
            (
                "user", """Extract all named entities. Google CEO Sundar Pichai visited India."""
            )
        ]
    }
)

print(response)

Using Cached spaCy Model.
{'messages': [HumanMessage(content='Extract all named entities. Google CEO Sundar Pichai visited India.', additional_kwargs={}, response_metadata={}, id='a52b4fe5-0aaa-42b4-9326-718c7a84e8e7'), AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'bn0en7wra', 'function': {'arguments': '{"text":"Google CEO Sundar Pichai visited India."}', 'name': 'ner_tool'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 23, 'prompt_tokens': 409, 'total_tokens': 432, 'completion_time': 0.067823478, 'completion_tokens_details': None, 'prompt_time': 0.06385422, 'prompt_tokens_details': None, 'queue_time': 0.162072147, 'total_time': 0.131677698}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_3272ea2d91', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019f6a6a-202f-7e83-8cc9-672f856bcdb3-0', tool_calls=[{'name': 'ner_tool', 'args': {'text': 'Google C

In [61]:
# Testing agent : 1
paper = """
Google DeepMind introduced Gemini, a multimodal large language model.
The research focuses on reasoning, planning and vision-language understanding.
The work was led by Demis Hassabis in London.
"""

response = agent.invoke(
    {
        "messages":[
            (
                "user",
                f"""
Analyze the following research text. Extract important keywords. Also identify all named entities.
Text: {paper}
"""
            )
        ]
    }
)

print(response)

Using Cached spaCy Model.
Using Cached KeyBERT Model.
{'messages': [HumanMessage(content='\nAnalyze the following research text. Extract important keywords. Also identify all named entities.\nText: \nGoogle DeepMind introduced Gemini, a multimodal large language model.\nThe research focuses on reasoning, planning and vision-language understanding.\nThe work was led by Demis Hassabis in London.\n\n', additional_kwargs={}, response_metadata={}, id='ce60a319-37c4-46b3-b2a8-eee8af13f695'), AIMessage(content='', additional_kwargs={'tool_calls': [{'id': '4t6b6nr6w', 'function': {'arguments': '{"text":"Google DeepMind introduced Gemini, a multimodal large language model. The research focuses on reasoning, planning and vision-language understanding. The work was led by Demis Hassabis in London."}', 'name': 'keyword_tool'}, 'type': 'function'}, {'id': '9sy5kn1sr', 'function': {'arguments': '{"text":"Google DeepMind introduced Gemini, a multimodal large language model. The research focuses on re

In [62]:
# TEsting agent : 2
paper = """
Large Language Models have become increasingly important in medical diagnosis.
Researchers from Stanford University and Microsoft developed a framework for clinical reasoning.
"""

response = agent.invoke(
    {
        "messages":[
            (
                "user",
                f"Analyze this research abstract:\n\n{paper}"
            )
        ]
    }
)

print(response)

Using Cached spaCy Model.
Using Cached KeyBERT Model.
Loading Summarizer...


model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Your max_length is set to 120, but your input_length is only 26. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=13)


{'messages': [HumanMessage(content='Analyze this research abstract:\n\n\nLarge Language Models have become increasingly important in medical diagnosis.\nResearchers from Stanford University and Microsoft developed a framework for clinical reasoning.\n', additional_kwargs={}, response_metadata={}, id='9c4dc8d7-a320-4b75-b048-5bf052baab61'), AIMessage(content='', additional_kwargs={'tool_calls': [{'id': '3k105tsg6', 'function': {'arguments': '{"text":"Large Language Models have become increasingly important in medical diagnosis. Researchers from Stanford University and Microsoft developed a framework for clinical reasoning."}', 'name': 'keyword_tool'}, 'type': 'function'}, {'id': 'hy483gz4g', 'function': {'arguments': '{"text":"Large Language Models have become increasingly important in medical diagnosis. Researchers from Stanford University and Microsoft developed a framework for clinical reasoning."}', 'name': 'ner_tool'}, 'type': 'function'}, {'id': 'k1rpa56e6', 'function': {'argument

In [63]:
# Testing agent : 3
sample_text = """
Google DeepMind recently introduced Gemini, a multimodal AI model capable of
reasoning across text, images, audio, and code.

The project was led by Demis Hassabis in London.

Researchers explained how deep learning and transformer architectures improve
large-scale reasoning and image understanding.

The research also discusses reinforcement learning and computer vision.
"""

response = agent.invoke(
    {
        "messages":[
            (
                "user",
                f"""
Summarize this text.

{sample_text}
"""
            )
        ]
    }
)

print(response["messages"][-1].content)

Your max_length is set to 120, but your input_length is only 67. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=33)


Google DeepMind recently introduced Gemini, a multimodal AI model capable of reasoning across text, images, audio, and code. Researchers explained how deep learning and transformer architectures improve large-scale reasoning and image understanding.


In [64]:
sample_text = """
Google DeepMind recently introduced Gemini, a multimodal AI model capable of
reasoning across text, images, audio, and code.

The project was led by Demis Hassabis in London.

Researchers explained how deep learning and transformer architectures improve
large-scale reasoning and image understanding.

The research also discusses reinforcement learning and computer vision.
"""

result = summarize_tool.invoke(
    {
        "text": sample_text
    }
)

print(result)

Your max_length is set to 120, but your input_length is only 79. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=39)


{'status': 'success', 'tool_name': 'summarize_tool', 'execution_time': 0.0, 'data': {'summary': 'Google DeepMind recently introduced Gemini, a multimodal AI model capable ofreasoning across text, images, audio, and code. The project was led by Demis Hassabis in London.'}, 'metadata': {'model': 'facebook/bart-large-cnn'}, 'error': None}


# A small chatbot  
where user can ask his query and agent will summarize, extract keywords or extract entity.

In [65]:
print("="*50)
print("InsightForge AI")
print("Type 'exit' to quit")
print("="*50)

while True:

    user_input = input("\nYou : ")

    if user_input.lower() == "exit":
        break

    response = agent.invoke(
        {
            "messages":[
                ("user", user_input)
            ]
        }
    )

    print("\nAI :")
    print(response["messages"][-1].content)

InsightForge AI
Type 'exit' to quit

You : extract keywords from this: Deep learning is a subset of Machine Learning and Artificial Intelligence.
Using Cached KeyBERT Model.

AI :
The extracted keywords are: deep learning, artificial intelligence, machine learning, deep, learning, learning artificial, learning subset, subset machine, intelligence, and machine.

You : can you identify a name in this : sunny leaone,  Ava Addams,  Aluara jenson,  Angela White
Using Cached spaCy Model.

AI :
The names mentioned in the text are:

* Ava Addams
* Aluara Jenson
* Angela White
* Sunny Leone

You : tell me if a fruit name in this phrase : I like potato , pea, mango , and chut
Using Cached KeyBERT Model.
Using Cached spaCy Model.

AI :
The fruit name in the phrase is "mango".

You : did you know what is the name of girls private part?

AI :
I'm not going to provide information on that topic. If you have any other questions or need information on a different subject, feel free to ask.

You : ok , 

Your max_length is set to 120, but your input_length is only 55. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=27)


Using Cached spaCy Model.
Using Cached KeyBERT Model.

AI :
The process name of producing young ones like themselves in the science chapter is reproduction. Reproduction is the process by which living organisms produce offspring that are similar to themselves. This can occur through various methods, including sexual reproduction, asexual reproduction, and budding.

You : what are the things a person should not search to protect or sstay away from 18+ porn sites?
Using Cached KeyBERT Model.
Using Cached spaCy Model.

AI :
To protect or stay away from 18+ porn sites, it's best to avoid searching for keywords such as "porn sites," "18 porn," "avoid searching," "sites keywords," "porn," "avoid 18," "keywords avoid," "searching," "sites," and "avoid." Additionally, being cautious when clicking on links or visiting unfamiliar websites can help prevent accidental exposure to such content.

You : what is the effect if one who is searches about these?


Your max_length is set to 120, but your input_length is only 8. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=4)


Using Cached KeyBERT Model.
Using Cached spaCy Model.

AI :
The effect of searching about these topics can have a significant effect on a person's mood. For more information on the effects of searching for these topics, visit CNN.com/soulmatestories.

You : what does mean by brazzers?
Using Cached spaCy Model.


Your max_length is set to 120, but your input_length is only 51. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=25)


Using Cached spaCy Model.

AI :
Brazzers is a popular adult entertainment website that features explicit content. It is known for its high-quality productions and is one of the most well-known brands in the adult entertainment industry.

You : what do you mean by adult entertainment?
Using Cached spaCy Model.
Using Cached KeyBERT Model.

AI :
Adult entertainment refers to a broad range of activities, performances, or services that are designed to entertain and engage adults, often with a focus on leisure, relaxation, and enjoyment. This can include various forms of entertainment such as music, dance, theater, film, and other performing arts, as well as activities like gaming, sports, and social events.

In some contexts, the term "adult entertainment" may also imply content or activities that are intended for mature audiences only, such as those with explicit themes, language, or content. However, it's essential to note that the term can have different connotations and interpretations 

In [77]:
from transformers import pipeline

summarizer = pipeline(
    "summarization",
    model="facebook/bart-large-cnn"
)

text = """
Deep learning has revolutionized computer vision by enabling neural networks
to automatically learn features from large datasets.
"""

input_words = len(text.split())

max_len = min(120, max(30, input_words // 2))
min_len = min(20, max_len - 5)

summary = summarizer(
    text,
    max_length=max_len,
    min_length=min_len,
    do_sample=False
)

print(result)

Your max_length is set to 30, but your input_length is only 25. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=12)


[{'summary_text': 'Deep learning has revolutionized computer vision by enabling neural networks to automatically learn features from large datasets.'}]


In [84]:
def format_analysis(summary=None, keywords=None, entities=None):
    """
    Format the outputs from different AI tools into a readable report.
    """

    report = "\n"
    report += "=" * 60 + "\n"
    report += "🧠 InsightForge AI Analysis Report\n"
    report += "=" * 60 + "\n\n"

    if summary:
        report += "📄 SUMMARY\n"
        report += "-" * 60 + "\n"
        report += summary + "\n\n"

    if keywords:
        report += "🔑 KEYWORDS\n"
        report += "-" * 60 + "\n"

        for keyword in keywords:
          report += (
            f"• {keyword['keyword'].title()} "
            f"(Score: {keyword['score']:.3f})\n"
        )

        report += "\n"

    if entities:
        report += "👤 NAMED ENTITIES\n"
        report += "-" * 60 + "\n"

        for entity in entities:

          LABEL_MAP = {
            "ORG": "Organization",
            "PERSON": "Person",
            "GPE": "Location",
            "LOC": "Location",
            "DATE": "Date",
            "TIME": "Time",
        }
          label = LABEL_MAP.get(
              entity["label"],
              entity["label"]
          )

          report += f"• {entity['text']} ({label})\n"

        report += "\n"

    return report

In [85]:
# ==========================================================
# INSIGHTFORGE AI ORCHESTRATOR
# ==========================================================
def analyze_document(text: str):

    summary = None
    keywords = None
    entities = None

    # -------- Summary --------
    summary_result = summarize_tool.invoke({"text": text})

    if summary_result["status"] == "success":
        summary = summary_result["data"]["summary"]

    # -------- Keywords --------
    keyword_result = keyword_tool.invoke({"text": text})

    if keyword_result["status"] == "success":
        keywords = keyword_result["data"]["keywords"]

    # -------- Entities --------
    ner_result = ner_tool.invoke({"text": text})

    if ner_result["status"] == "success":
        entities = ner_result["data"]["entities"]

    report = format_analysis(
        summary,
        keywords,
        entities
    )

    return report

In [86]:
#Testing : 4
sample_text = """
Google DeepMind recently introduced Gemini, a multimodal AI model capable of reasoning across text, images, audio, and code.

The project was led by Demis Hassabis in London.

Researchers explained how deep learning and transformer architectures improve large-scale reasoning and image understanding.

The research also discusses reinforcement learning and computer vision.
"""
report = analyze_document(sample_text)

print(report)

Using Cached KeyBERT Model.
Using Cached spaCy Model.

🧠 InsightForge AI Analysis Report

📄 SUMMARY
------------------------------------------------------------
Google DeepMind recently introduced Gemini, a multimodal AI model capable of reasoning across text, images, audio, and code. The

🔑 KEYWORDS
------------------------------------------------------------
• Google Deepmind (Score: 0.735)
• Deepmind (Score: 0.674)
• Deepmind Recently (Score: 0.643)
• Multimodal Ai (Score: 0.611)
• Deep Learning (Score: 0.520)
• Gemini Multimodal (Score: 0.506)
• Ai (Score: 0.505)
• Reasoning Image (Score: 0.497)
• Ai Model (Score: 0.492)
• Multimodal (Score: 0.489)

👤 NAMED ENTITIES
------------------------------------------------------------
• Google DeepMind (Organization)
• Gemini (Location)
• Demis Hassabis (Organization)
• London (Location)


